# Ingest AESO API data into bronze (watermark + lookback + MERGE)

Extends the historical CSV past July 2025 by pulling the public AESO API and
MERGE-ing into Delta **bronze** tables (Change Data Feed on at creation):

| table | grain | API source |
|---|---|---|
| `bronze_pool_price` | `begin_datetime_utc` | Pool Price Report |
| `bronze_ail` | `begin_datetime_utc` | Actual Forecast Report (AIL) |
| `bronze_generation` | `begin_datetime_utc, asset_id` | Metered Volume (GENCO/IPP) |
| `bronze_interchange` | `begin_datetime_utc, path` | Metered Volume (EXPORTER/IMPORTER), aggregated per path |

One idempotent code path: a `watermark` reads each table's max timestamp (falling
back to `GENESIS` on first run), re-pulls a trailing `LOOKBACK` window so AESO
settlement **revisions** overwrite via the MERGE, and ingests up to today in
memory-bounded windows. See `api_scratch/ENDPOINTS.md` for endpoint quirks
(MPT-based inclusive dates; metered-volume `endDate` is EXCLUSIVE, ~16-day cap).


In [ ]:
dbutils.widgets.text("catalog_name", "classic_stable_ptbvhz_ip")
dbutils.widgets.text("schema_name", "alberta_energy")
dbutils.widgets.text(
    "genesis", "2025-08-01"
)  # first hour after the CSV ends (MPT date)
dbutils.widgets.text("secret_scope", "aeso")
dbutils.widgets.text("secret_key", "api")
dbutils.widgets.text(
    "max_window_days", ""
)  # optional cap on the fetch+MERGE window (split a heavy first backfill)
dbutils.widgets.text(
    "force_lookback_days", ""
)  # optional: re-pull this many days for ALL sources (monthly resettlement)

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
genesis = dbutils.widgets.get("genesis")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")

In [ ]:
import re
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

import requests
from pyspark.sql import functions as F

BASE = "https://apimgw.aeso.ca/public"
MPT = ZoneInfo("America/Edmonton")  # AESO date params are Mountain-based

API_KEY = dbutils.secrets.get(scope=secret_scope, key=secret_key)
SESSION = requests.Session()
SESSION.headers.update({"API-KEY": API_KEY, "Cache-Control": "no-cache"})

# trailing window re-pulled each run so AESO settlement revisions overwrite via MERGE
# (generation settles slowest -> longer lookback)
LOOKBACK_DAYS = {"pool_price": 3, "ail": 3, "generation": 14, "interchange": 14}
# fetch + MERGE window per source: bounds driver memory and respects each endpoint's max range
WINDOW_DAYS = {"pool_price": 366, "ail": 366, "generation": 15, "interchange": 15}
_ovr = dbutils.widgets.get("max_window_days").strip()
if _ovr:
    WINDOW_DAYS = {k: min(v, int(_ovr)) for k, v in WINDOW_DAYS.items()}

_flb = dbutils.widgets.get("force_lookback_days").strip()
FORCE_LOOKBACK = (
    int(_flb) if _flb else None
)  # overrides LOOKBACK_DAYS for every source when set

GENESIS = date.fromisoformat(genesis)
TODAY_MPT = datetime.now(MPT).date()
print(
    f"GENESIS={GENESIS}  TODAY_MPT={TODAY_MPT}  WINDOW_DAYS={WINDOW_DAYS}  FORCE_LOOKBACK={FORCE_LOOKBACK}"
)

## HTTP helpers

In [ ]:
def _get(path, start, end):
    """GET one window. Returns the `return` payload."""
    params = {"startDate": start.isoformat(), "endDate": end.isoformat()}
    r = SESSION.get(BASE + path, params=params, timeout=300)
    r.raise_for_status()
    return r.json()["return"]


def _chunks(start, end, span_days, end_exclusive):
    """Yield (chunk_start, api_end) covering [start, end] inclusive.

    span_days = max days of data the endpoint returns per call.
    end_exclusive: True if the API's endDate is exclusive (metered volume).
    """
    cur = start
    while cur <= end:
        last = min(
            cur + timedelta(days=span_days - 1), end
        )  # last day we want, inclusive
        api_end = last + timedelta(days=1) if end_exclusive else last
        yield cur, api_end
        cur = last + timedelta(days=1)

## Fetchers (live-tested in `api_scratch/aeso_backfill.py`)

In [ ]:
def fetch_pool_price(start, end):
    rows = []
    for s, e in _chunks(start, end, 366, end_exclusive=False):
        for rec in _get("/poolprice-api/v1.1/price/poolPrice", s, e)[
            "Pool Price Report"
        ]:
            rows.append(
                {
                    "begin_datetime_utc": rec["begin_datetime_utc"],
                    "begin_datetime_mpt": rec["begin_datetime_mpt"],
                    "pool_price": rec["pool_price"],
                    "forecast_pool_price": rec["forecast_pool_price"],
                    "rolling_30day_avg": rec["rolling_30day_avg"],
                }
            )
    return rows


def fetch_ail(start, end):
    rows = []
    for s, e in _chunks(start, end, 366, end_exclusive=False):
        for rec in _get("/actualforecast-api/v1/load/albertaInternalLoad", s, e)[
            "Actual Forecast Report"
        ]:
            rows.append(
                {
                    "begin_datetime_utc": rec["begin_datetime_utc"],
                    "begin_datetime_mpt": rec["begin_datetime_mpt"],
                    "alberta_internal_load": rec["alberta_internal_load"],
                    "forecast_alberta_internal_load": rec[
                        "forecast_alberta_internal_load"
                    ],
                }
            )
    return rows


def fetch_generation(start, end, asset_classes=("GENCO", "IPP")):
    """Per-asset hourly metered volume (long). endDate EXCLUSIVE, max (endDate - startDate) = 15."""
    keep = set(asset_classes)
    rows = []
    for s, e in _chunks(start, end, 15, end_exclusive=True):
        for part in _get("/meteredvolume-api/v1/meteredvolume/details", s, e):
            for asset in part["asset_list"]:
                if asset["asset_class"] not in keep:
                    continue
                for mv in asset["metered_volume_list"]:
                    rows.append(
                        {
                            "begin_datetime_utc": mv["begin_date_utc"],
                            "begin_datetime_mpt": mv["begin_date_mpt"],
                            "pool_participant_id": part["pool_participant_ID"],
                            "asset_id": asset["asset_ID"],
                            "asset_class": asset["asset_class"],
                            "metered_volume": mv["metered_volume"],
                        }
                    )
    return rows

## Interchange (interties)

The Metered Volume report exposes per-asset EXPORTER/IMPORTER meters but no path
column. We map each intertie asset to a **path** (BC / SK / MT) from its name in
the Asset List, with an asset-ID-suffix fallback, then aggregate to per-path
import/export totals (so the pipeline needs no streaming aggregation). Validated
to classify 164/167 intertie assets; test units (TSTE/TSTI) and the PWSR spinning-
reserve unit are intentionally dropped. Note: the CSV only had BC + SK, so the
**MT (Montana)** path appears only from the API era onward.

In [ ]:
def _intertie_path_map():
    """asset_ID -> path (BC|SK|MT|None) from the Asset List names."""
    r = SESSION.get(BASE + "/assetlist-api/v1/assetlist", timeout=300)
    r.raise_for_status()
    names = {a["asset_ID"]: (a.get("asset_name") or "") for a in r.json()["return"]}

    def classify(aid, name):
        u = name.upper()
        if "RESERVE" in u:  # not a directional intertie path
            return None
        if "MONTANA" in u or re.search(r"\bMT\b", u):
            return "MT"
        if "SASK" in u or "SPC" in u or re.search(r"\bSK\b", u):
            return "SK"
        if "BCH" in u or re.search(r"\bBC\b", u):
            return "BC"
        # fallback: asset_ID suffix convention (xxBC/xxSK/xxMT import, xXB/xXS/xXM export)
        return {
            "BC": "BC",
            "SK": "SK",
            "MT": "MT",
            "XB": "BC",
            "XS": "SK",
            "XM": "MT",
        }.get(aid[-2:].upper())

    return {aid: classify(aid, nm) for aid, nm in names.items()}


PATH_MAP = _intertie_path_map()


def fetch_interchange(start, end):
    """Per-asset intertie meters tagged with path + direction (aggregated later)."""
    rows = []
    for s, e in _chunks(start, end, 15, end_exclusive=True):
        for part in _get("/meteredvolume-api/v1/meteredvolume/details", s, e):
            for asset in part["asset_list"]:
                cls = asset["asset_class"]
                if cls not in ("EXPORTER", "IMPORTER"):
                    continue
                path = PATH_MAP.get(asset["asset_ID"])
                if path is None:  # test/reserve/unmapped
                    continue
                direction = "import" if cls == "IMPORTER" else "export"
                for mv in asset["metered_volume_list"]:
                    rows.append(
                        {
                            "begin_datetime_utc": mv["begin_date_utc"],
                            "begin_datetime_mpt": mv["begin_date_mpt"],
                            "path": path,
                            "direction": direction,
                            "metered_volume": mv["metered_volume"],
                        }
                    )
    return rows

## Bronze table specs + DDL

Explicit `CREATE TABLE` with table + column comments and Change Data Feed in `TBLPROPERTIES` (on at creation, so backfill rows land in the change feed for the SDP pipeline).

In [ ]:
TABLES = {
    "pool_price": {
        "name": "bronze_pool_price",
        "keys": ["begin_datetime_utc"],
        "comment": "AESO hourly pool price (actual, hour-ahead forecast, 30-day average) from the public Pool Price Report API. Bronze landing that extends the historical CSV past July 2025; upserted via MERGE with a trailing lookback so settlement revisions overwrite. Change Data Feed enabled for the downstream SDP pipeline.",
        "columns": [
            (
                "begin_datetime_utc",
                "TIMESTAMP",
                "Hour-beginning timestamp, UTC wall clock. Natural key.",
            ),
            (
                "begin_datetime_mpt",
                "TIMESTAMP",
                "Hour-beginning timestamp, Mountain Prevailing Time.",
            ),
            ("pool_price", "DOUBLE", "Actual Alberta pool price (CAD per MWh)."),
            (
                "forecast_pool_price",
                "DOUBLE",
                "Hour-ahead forecast pool price (CAD per MWh).",
            ),
            (
                "rolling_30day_avg",
                "DOUBLE",
                "Rolling 30-day average pool price (CAD per MWh).",
            ),
        ],
    },
    "ail": {
        "name": "bronze_ail",
        "keys": ["begin_datetime_utc"],
        "comment": "AESO hourly Alberta Internal Load (AIL), actual and forecast, from the public Actual Forecast Report API. Bronze landing that extends the historical CSV past July 2025; upserted via MERGE with a trailing lookback. Change Data Feed enabled for the downstream SDP pipeline.",
        "columns": [
            (
                "begin_datetime_utc",
                "TIMESTAMP",
                "Hour-beginning timestamp, UTC wall clock. Natural key.",
            ),
            (
                "begin_datetime_mpt",
                "TIMESTAMP",
                "Hour-beginning timestamp, Mountain Prevailing Time.",
            ),
            ("alberta_internal_load", "DOUBLE", "Actual Alberta Internal Load (MW)."),
            (
                "forecast_alberta_internal_load",
                "DOUBLE",
                "Forecast Alberta Internal Load (MW).",
            ),
        ],
    },
    "generation": {
        "name": "bronze_generation",
        "keys": ["begin_datetime_utc", "asset_id"],
        "comment": "AESO per-asset hourly metered generation volume (GENCO and IPP asset classes) from the public Metered Volume Report API. One row per generating asset per hour; the long-format equivalent of the wide per-asset generation columns in the historical CSV. Upserted via MERGE with a 14-day lookback (generation settles slowest). Change Data Feed enabled for the downstream SDP pipeline.",
        "columns": [
            (
                "begin_datetime_utc",
                "TIMESTAMP",
                "Hour-beginning timestamp, UTC wall clock. Natural key part.",
            ),
            (
                "begin_datetime_mpt",
                "TIMESTAMP",
                "Hour-beginning timestamp, Mountain Prevailing Time.",
            ),
            (
                "pool_participant_id",
                "STRING",
                "AESO pool participant ID(s) reporting the asset; comma-separated when more than one participant reports it in the hour (e.g. during an ownership transition).",
            ),
            (
                "asset_id",
                "STRING",
                "AESO asset ID. Natural key part; join to the Asset List for name and fuel type.",
            ),
            (
                "asset_class",
                "STRING",
                "Asset class as reported by AESO (GENCO or IPP after the ingest filter).",
            ),
            (
                "metered_volume",
                "DOUBLE",
                "Hourly net metered generation (MWh), summed across all participants reporting the asset in the hour.",
            ),
        ],
    },
    "interchange": {
        "name": "bronze_interchange",
        "keys": ["begin_datetime_utc", "path"],
        "comment": "AESO hourly intertie flows by path (BC, SK, MT), aggregated from the per-asset EXPORTER and IMPORTER metered volumes in the public Metered Volume Report API. The historical CSV only exposed BC and SK, so the Montana (MT) path appears only from the API era onward. Upserted via MERGE with a 14-day lookback. Change Data Feed enabled for the downstream SDP pipeline.",
        "columns": [
            (
                "begin_datetime_utc",
                "TIMESTAMP",
                "Hour-beginning timestamp, UTC wall clock. Natural key part.",
            ),
            (
                "begin_datetime_mpt",
                "TIMESTAMP",
                "Hour-beginning timestamp, Mountain Prevailing Time.",
            ),
            (
                "path",
                "STRING",
                "Intertie path: BC (British Columbia), SK (Saskatchewan), or MT (Montana). Natural key part.",
            ),
            (
                "import_mw",
                "DOUBLE",
                "Total hourly import volume into Alberta on this path (MWh).",
            ),
            (
                "export_mw",
                "DOUBLE",
                "Total hourly export volume out of Alberta on this path (MWh).",
            ),
            (
                "net_export_mw",
                "DOUBLE",
                "Net export on this path (export_mw minus import_mw); positive means net export out of Alberta (MWh).",
            ),
        ],
    },
}


def ensure_table(source):
    spec = TABLES[source]
    fqtn = f"{catalog_name}.{schema_name}.{spec['name']}"
    cols = ",\n            ".join(
        f"`{n}` {t} COMMENT '{c}'" for n, t, c in spec["columns"]
    )
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {fqtn} (
            {cols}
        )
        USING DELTA
        COMMENT '{spec["comment"]}'
        TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)
    return fqtn

## Typed DataFrame builders

Column order matches each table spec. API timestamps are UTC-labelled wall clock (`yyyy-MM-dd HH:mm`), parsed naively to match how the CSV's `Date_Begin_GMT` is parsed in the pipeline so the seam joins.

In [ ]:
def _ts(c):
    return F.to_timestamp(F.col(c), "yyyy-MM-dd HH:mm")


def _dbl(c):
    # AESO returns "" for some numeric fields; try_cast yields NULL instead of erroring (ANSI)
    return F.expr(f"try_cast(`{c}` AS double)")


def df_pool_price(rows):
    return spark.createDataFrame(rows).select(
        _ts("begin_datetime_utc").alias("begin_datetime_utc"),
        _ts("begin_datetime_mpt").alias("begin_datetime_mpt"),
        _dbl("pool_price").alias("pool_price"),
        _dbl("forecast_pool_price").alias("forecast_pool_price"),
        _dbl("rolling_30day_avg").alias("rolling_30day_avg"),
    )


def df_ail(rows):
    return spark.createDataFrame(rows).select(
        _ts("begin_datetime_utc").alias("begin_datetime_utc"),
        _ts("begin_datetime_mpt").alias("begin_datetime_mpt"),
        _dbl("alberta_internal_load").alias("alberta_internal_load"),
        _dbl("forecast_alberta_internal_load").alias("forecast_alberta_internal_load"),
    )


def df_generation(rows):
    # an asset can be reported under >1 participant (ownership transitions); sum to the
    # (hour, asset) grain so metered_volume is the asset's total net generation and the
    # MERGE key (begin_datetime_utc, asset_id) is unique
    return (
        spark.createDataFrame(rows)
        .groupBy("begin_datetime_utc", "begin_datetime_mpt", "asset_id")
        .agg(
            F.concat_ws(",", F.sort_array(F.collect_set("pool_participant_id"))).alias(
                "pool_participant_id"
            ),
            F.max("asset_class").alias("asset_class"),
            F.sum(_dbl("metered_volume")).alias("metered_volume"),
        )
        .select(
            _ts("begin_datetime_utc").alias("begin_datetime_utc"),
            _ts("begin_datetime_mpt").alias("begin_datetime_mpt"),
            "pool_participant_id",
            "asset_id",
            "asset_class",
            "metered_volume",
        )
    )


def df_interchange(rows):
    mv = _dbl("metered_volume")
    agg = (
        spark.createDataFrame(rows)
        .groupBy("begin_datetime_utc", "begin_datetime_mpt", "path")
        .agg(
            F.sum(F.when(F.col("direction") == "import", mv).otherwise(0.0)).alias(
                "import_mw"
            ),
            F.sum(F.when(F.col("direction") == "export", mv).otherwise(0.0)).alias(
                "export_mw"
            ),
        )
    )
    return agg.select(
        _ts("begin_datetime_utc").alias("begin_datetime_utc"),
        _ts("begin_datetime_mpt").alias("begin_datetime_mpt"),
        F.col("path"),
        F.col("import_mw").cast("double").alias("import_mw"),
        F.col("export_mw").cast("double").alias("export_mw"),
        (F.col("export_mw") - F.col("import_mw")).cast("double").alias("net_export_mw"),
    )

## Watermark + MERGE + driver

In [ ]:
def watermark(fqtn):
    """Latest ingested MPT date, or GENESIS on first run."""
    m = spark.sql(f"SELECT max(begin_datetime_mpt) AS m FROM {fqtn}").first().m
    return m.date() if m is not None else GENESIS


def merge_delta(fqtn, df, keys):
    # Only rewrite a matched row when a non-key value actually changed (null-safe `<=>`).
    # The trailing lookback re-pulls a fixed window every run, but most rows are identical
    # to what's already in bronze; an unconditional UPDATE SET * would rewrite them all and
    # emit them into the Change Data Feed, churning ~250k generation rows downstream each run.
    # Gating on a real change keeps the CDF (and the SDP pipeline) to genuinely-revised rows.
    df.createOrReplaceTempView("_updates")
    on = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    val_cols = [c for c in df.columns if c not in keys]
    changed = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in val_cols)
    matched = f"WHEN MATCHED AND ({changed}) THEN UPDATE SET *\n" if val_cols else ""
    spark.sql(f"""
        MERGE INTO {fqtn} t USING _updates s ON {on}
        {matched}WHEN NOT MATCHED THEN INSERT *
    """)


def ingest(source, fetch_fn, df_fn):
    spec = TABLES[source]
    fqtn = ensure_table(source)
    keys = spec["keys"]
    lookback = FORCE_LOOKBACK if FORCE_LOOKBACK is not None else LOOKBACK_DAYS[source]
    lo = max(watermark(fqtn) - timedelta(days=lookback), GENESIS)
    hi = TODAY_MPT
    win = WINDOW_DAYS[source]
    print(f"[{source}] {fqtn}  window {lo}..{hi}")
    cur = lo
    while cur <= hi:
        wend = min(cur + timedelta(days=win - 1), hi)
        rows = fetch_fn(cur, wend)
        if rows:
            merge_delta(fqtn, df_fn(rows), keys)
        print(f"  [{source}] {cur}..{wend}: {len(rows)} source rows")
        cur = wend + timedelta(days=1)
    print(f"[{source}] done")

## Run

In [ ]:
ingest("pool_price", fetch_pool_price, df_pool_price)
ingest("ail", fetch_ail, df_ail)
ingest("generation", fetch_generation, df_generation)
ingest("interchange", fetch_interchange, df_interchange)